In [1]:
# ==============================================================================
# PIPELINE 5: Final Dataset Cleaning (Logical Errors & Outliers)
# ==============================================================================
import pandas as pd
import numpy as np
from src.profiling import generate_minimal_report, generate_full_sample_report
from src.preparation.merged_preparation import (
    remove_logical_errors,
    remove_extreme_outliers,
    impute_domain_specific_missing_values,
    winsorize_features,
    drop_missing_targets
)

pd.set_option('display.max_columns', None)

print("Loading Golden Dataset from Pipeline 4...")
df_ml = pd.read_parquet("data/prepared/ted_combined.parquet")
print(f"Initial shape: {df_ml.shape}")

Loading Golden Dataset from Pipeline 4...
Initial shape: (4623976, 24)


In [2]:
# ------------------------------------------------------------------------------
# STEP 1: Target Integrity & Logical Cleaning
# ------------------------------------------------------------------------------

# Dropping records without ground truth to ensure high-quality training labels.
critical_targets = ['TARGET_AWARD_VALUE_EUR', 'LOT_AWARD_VALUE_EUR', 'NUMBER_OF_TENDERS']
df_ml = drop_missing_targets(df_ml, target_columns=critical_targets)

# Purging impossible data points (negative time or money) before statistical analysis.
logical_check_cols = [
    'PREPARATION_DAYS', 'DURATION', 'TARGET_AWARD_VALUE_EUR', 
    'LOT_AWARD_VALUE_EUR', 'ESTIMATED_VALUE_EUR', 'NUMBER_OF_TENDERS'
]
df_ml = remove_logical_errors(df_ml, columns=logical_check_cols, min_val=0.0)

Dropping rows with missing critical target variables: ['TARGET_AWARD_VALUE_EUR', 'LOT_AWARD_VALUE_EUR', 'NUMBER_OF_TENDERS']...
 -> Dropped 2,192,352 unusable rows.
Removing logical errors (Dropping values < 0.0)...
 -> Dropped 10 rows with impossible negative values.


In [3]:
# ------------------------------------------------------------------------------
# STEP 2: Domain-Specific Imputation (Missing Not At Random)
# ------------------------------------------------------------------------------
# Applying safe constants (0 or 'MISSING') before the ML splits guarantees 
# that these meaningful categories are correctly represented in EDA heatmaps.
df_ml = impute_domain_specific_missing_values(df_ml)

Applying domain-specific logic to missing values...
  -> B_EU_FUNDS: Filled 12,691 NaNs with 0 (No).
  -> B_RECURRENT_PROCUREMENT: Filled 46,463 NaNs with 0 (No).
  -> CRIT_CODE: Filled 425,483 NaNs with 'MISSING'.
  -> TOP_TYPE: Filled 4,007 NaNs with mode 'OPE'.


In [4]:
# ------------------------------------------------------------------------------
# STEP 2: Mathematical Outlier Detection (Group-Specific Methods)
# ------------------------------------------------------------------------------

# 2.1 TRIMMING (Dropping corrupted target values)
# Targets involving finance or competition (Tender Counts) scale across multiple
# magnitudes. Log-IQR is required to differentiate between high-value projects 
# and actual data entry errors.
target_log_cols = ['TARGET_AWARD_VALUE_EUR', 'LOT_AWARD_VALUE_EUR', 'NUMBER_OF_TENDERS']
df_ml = remove_extreme_outliers(df_ml, columns=target_log_cols, k_factor=3.0, method='log')

# 2.2 WINSORIZING (Capping input features)
# GROUP A/B: Finance and Counts (Heavy-Tailed) -> Log-IQR Method
# Capping prevents exponential features from dominating the model optimization process.
input_log_cols = ['ESTIMATED_VALUE_EUR', 'LOTS_NUMBER']
df_ml = winsorize_features(df_ml, columns=input_log_cols, k_factor=3.0, method='log')

# GROUP C: Time and Regions (Linear Scaling) -> Standard-IQR Method
# These features lack the power-law distribution of finance; thus, a linear 
# Tukey boundary is mathematically appropriate.
input_standard_cols = ['DURATION', 'PREPARATION_DAYS', 'NUTS_REGION_COUNT']
df_ml = winsorize_features(df_ml, columns=input_standard_cols, k_factor=3.0, method='standard')

Applying mathematical trimming (Log-IQR, k=3.0)...
 -> Dropped 7,153 rows exceeding mathematical Log-IQR boundaries.
Winsorizing features (Capping via Log-IQR, k=3.0)...
  -> ESTIMATED_VALUE_EUR: Capped 64 values at mathematically derived max (5,070,030,253.73).
Winsorizing features (Capping via Standard-IQR, k=3.0)...
  -> DURATION: Capped 8,780 values at mathematically derived max (108.00).
  -> PREPARATION_DAYS: Capped 51,770 values at mathematically derived max (57.00).
  -> NUTS_REGION_COUNT: Capped 71,657 values at mathematically derived max (1.00).


In [5]:
# ------------------------------------------------------------------------------
# STEP 2: Export Prepared Dataset
# ------------------------------------------------------------------------------
output_path = "data/prepared/ted_prepared.parquet"
df_ml.to_parquet(output_path, index=False)

generate_minimal_report(
    df=df_ml,
    output_path="reports/merged/ted_prepared_minimal.html",
    title="TED Prepared (Clean, Raw Scale)"
)

# Generate a full report for a sample of the merged data
profiling_subset = df_ml.drop(columns=['ID_NOTICE', 'ID_LOT'], errors='ignore')
skewed_cols = ['NUMBER_OF_TENDERS', 'LOTS_NUMBER', 'DURATION', 'NUTS_LEVEL', 'PREPARATION_DAYS', 'LOT_AWARD_VALUE_EUR', 'TARGET_AWARD_VALUE_EUR', 'ESTIMATED_VALUE_EUR']
for col in skewed_cols:
    if col in profiling_subset.columns:
        profiling_subset[f"{col}_LOG1P"] = np.log1p(profiling_subset[col])
        # Drop the raw unscaled column from the report view to prevent crashes
        profiling_subset = profiling_subset.drop(columns=[col])

generate_full_sample_report(
    df=profiling_subset,
    output_path="reports/merged/ted_prepared_full_sample.html",
    title="TED Prepared (full; 250k sample)",
    sample_size=250000
)

Generating minimal report: 'TED Prepared (Clean, Raw Scale)' ...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 24/24 [00:00<00:00, 30.18it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 22/22 [00:00<00:00, 234.45it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

-> Full report successfully saved to: reports/merged/ted_prepared_full_sample.html

